
<div class="alert alert-block alert-info">  
    
## Import & Connections
</div>

In [1]:
import numpy as np
import pandas as pd
import hashlib
import glob
import time
import os
import re
import matplotlib.pyplot as plt

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_rows', 300)
pd.set_option('display.max_columns', 50)

In [3]:
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [4]:
start_date = '20250130'
end_date = '20250205'
response_type = 'ONLINE_SALES'
pagename = 'PostOrderScreen'
campaign_name = 'Exp3_MaxBanner_Zepto'
format = 'maxBanner'

In [5]:
print(os.getcwd())
local_dataset = '/Users/E2074/local-datasets/customer/ads-monetisation/page-format-experiment/max-banner'
print(local_dataset)

/Users/E2074/analytics/customer/Ads-Monetisation/page-format-experiment
/Users/E2074/local-datasets/customer/ads-monetisation/page-format-experiment/max-banner


<div class="alert alert-block alert-warning">  
    
## Datasets & Functions
</div>

In [6]:
start = datetime.strptime(start_date, '%Y%m%d')
end = datetime.strptime(end_date, '%Y%m%d')

# Generate the list of dates
date_range = [(start + timedelta(days=i)).strftime('%Y%m%d') for i in range((end - start).days + 1)]
print(date_range)

['20250130', '20250131', '20250201', '20250202', '20250203', '20250204', '20250205']


In [7]:
def get_customer_metadata(date_range):

    all_df = []
    
    for date in date_range:
    
        customer_metadata = f"""
        
            with response as (
            
            select
                distinct
                yyyymmdd,
                json_extract_scalar(metadata1, '$.campaignName') AS campaignName,
                json_extract_scalar(metadata1, '$.slotUnitId') AS slotUnitId,
                city,
                os,
                userid,
                service,
                orderstatus,
                -- totalcardcount,
                -- cardposition,
                coalesce(aduuid, adid) response_id
            from 
                canonical.iceberg_log_telemetry_ads_impressions_immutable
            where 
                yyyymmdd = '{date}'
                and os = 'android'
                and responsetype = '{response_type}'
                and eventname = 'Ad_Response'
                and pagename = '{pagename}'
                and format = '{format}'
                and regexp_like(metadata1, '{campaign_name}')
            ),
            
            impression as (
            
            select
                distinct
                yyyymmdd,
                userid,
                coalesce(aduuid, adid) impression_id
            from 
                canonical.iceberg_log_telemetry_ads_impressions_immutable
            where 
                yyyymmdd = '{date}'
                and os = 'android'
                and responsetype = '{response_type}'
                and eventname = 'Ad_Impression'
                and pagename = '{pagename}'
                and format = '{format}'
                and regexp_like(metadata1, '{campaign_name}')
            ),
            
            viewable_impression as (
            
            select
                distinct
                yyyymmdd,
                userid,
                coalesce(aduuid, adid) viewable_impression_id
            from 
                canonical.iceberg_log_telemetry_ads_impressions_immutable
            where 
                yyyymmdd = '{date}'
                and os = 'android'
                and responsetype = '{response_type}'
                and eventname = 'Ad_Viewable_Impression'
                and pagename = '{pagename}'
                and format = '{format}'
                and regexp_like(metadata1, '{campaign_name}')
            ),
            
            click as (
            
            select
                distinct
                yyyymmdd,
                userid,
                coalesce(aduuid, adid) click_id
            from 
                canonical.iceberg_log_telemetry_ads_impressions_immutable
            where 
                yyyymmdd = '{date}'
                and os = 'android'
                and responsetype = '{response_type}'
                and eventname = 'Ad_Click'
                and pagename = '{pagename}'
                and format = '{format}'
            )
            
            select
                a.*,
                impression_id,
                coalesce(click_id, viewable_impression_id) viewable_impression_id,
                click_id
            from 
                response a
            left join 
                impression b
                on a.yyyymmdd = b.yyyymmdd
                and a.userid = b.userid
                and a.response_id = b.impression_id
            left join 
                viewable_impression c
                on b.yyyymmdd = c.yyyymmdd
                and b.userid = c.userid
                and b.impression_id = c.viewable_impression_id
            left join 
                click d
                on b.yyyymmdd = d.yyyymmdd
                and b.userid = d.userid
                and b.impression_id = d.click_id
        """
        
        df = pd.read_sql(customer_metadata, connection)
        all_df.append(df)
        df.to_parquet(local_dataset + '/data_dump/{}_{}.parquet'.format(campaign_name, date), index=False)
    
    # Concatenate all DataFrames
    final_df = pd.concat(all_df, ignore_index=True)

    return final_df


# df_data = get_customer_metadata(date_range)

In [8]:
def get_customer_segment():    
    
    customer_segment = f"""
    
        with response as (

            select
                distinct
                userid
            from 
                canonical.iceberg_log_telemetry_ads_impressions_immutable
            where 
                yyyymmdd >= '{start_date}'
                and yyyymmdd <= '{end_date}'
                and os = 'android'
                and responsetype = '{response_type}'
                and eventname = 'Ad_Response'
                and pagename = '{pagename}'
                and format = '{format}'
                and regexp_like(metadata1, '{campaign_name}')
            ),

            segment as (
    
            select
                customer_id,
                gender_tag,
                taxi_ltr_segment,
                taxi_retention_segments,
                taxi_regularity_segment,
                taxi_fe_regularity_segment,
                taxi_income_segment,
                taxi_service_affinity,
                taxi_gc_tag
            from 
                datasets.iallocator_customer_segments
            where 
                run_date = '2025-01-29'
            )
            
            select
                userid,
                coalesce(gender_tag, 'UNKNOWN') gender,
                coalesce(taxi_ltr_segment, 'NEW') ltr_segment,
                coalesce(taxi_retention_segments, 'UNKNOWN') retention_segments,
                coalesce(taxi_regularity_segment, 'UNKNOWN') regularity_segment,
                coalesce(taxi_fe_regularity_segment, 'UNKNOWN') fe_regularity_segment,
                coalesce(taxi_income_segment, 'UNKNOWN') income_segment,
                coalesce(taxi_service_affinity, 'UNKNOWN') service_affinity,
                coalesce(taxi_gc_tag, 'UNKNOWN') gc_tag
            from 
                response
            left join 
                segment
                on userid = customer_id
            
    """
    
    df = pd.read_sql(customer_segment, connection)
    df.to_parquet(local_dataset + '/Exp3_customer_segment.parquet', index=False)

    return df

# df_data = get_customer_segment()


<div class="alert alert-block alert-info">  
    
## Load data
</div>

In [9]:
def get_customer_metadata(prefix):
    
    parquet_files = glob.glob(os.path.join(local_dataset + '/data_dump/', f'{prefix}*.parquet'))
    
    customer_metadata = pd.concat([pd.read_parquet(file) for file in parquet_files], ignore_index=True)
    customer_segment = pd.read_parquet(local_dataset + f'/{prefix}_customer_segment.parquet')
    
    df = pd.merge(customer_metadata, customer_segment, how='left', on = 'userid')
    return df

In [10]:
df_exp1 = get_customer_metadata('Exp1')
df_exp2 = get_customer_metadata('Exp2')
df_exp3 = get_customer_metadata('Exp3')

In [11]:
df_exp1['Experiment'] = 'Experiment-1'
df_exp2['Experiment'] = 'Experiment-2'
df_exp3['Experiment'] = 'Experiment-3'

In [12]:
frames = [df_exp1, df_exp2, df_exp3]
df_data = pd.concat(frames)

In [13]:
df_data.head(2)

,yyyymmdd,campaignName,slotUnitId,city,os,userid,service,orderstatus,response_id,impression_id,viewable_impression_id,click_id,gender,ltr_segment,retention_segments,regularity_segment,fe_regularity_segment,income_segment,service_affinity,gc_tag,Experiment
0,20250115,Exp1_MaxBanner_Zepto,PostOrderArrived3,Bangalore,android,5e5f1abf32e3140a0beaf16d,Auto,arrived,b57dbc1c-b86e-4519-a460-30a5478d9cd65e5f1abf32...,b57dbc1c-b86e-4519-a460-30a5478d9cd65e5f1abf32...,b57dbc1c-b86e-4519-a460-30a5478d9cd65e5f1abf32...,None,FEMALE,PHH,PLATINUM,DAILY,DAILY,MEDIUM_INCOME,ONLY_AUTO,UNKNOWN,Experiment-1
1,20250115,Exp1_MaxBanner_Zepto,PostOrderOnTheWay3,Bangalore,android,676c0d81a353a7da26899fd1,Auto,onTheWay,b8e7c852-d0f3-488f-af24-f3f415854597676c0d81a3...,b8e7c852-d0f3-488f-af24-f3f415854597676c0d81a3...,None,None,MALE,PHH,GOLD,BI_WEEKLY,MONTHLY,UNKNOWN,ONLY_LINK,UNKNOWN,Experiment-1


In [14]:
df_data['yyyymmdd'].value_counts().sort_index(ascending=True)

yyyymmdd
20250115    219413
20250116    232757
20250117    252570
20250118         9
20250119        29
20250120    278302
20250121    281593
20250122    288297
20250123    263244
20250124    269672
20250125    257098
20250126    261591
20250127    267431
20250128    275983
20250129    278864
20250130    289245
20250131    291791
20250201    270462
20250202    274591
20250203    285253
20250204    292258
20250205    302847
Name: count, dtype: int64

<div class="alert alert-block alert-warning">  
    
## Funnel Func
</div>

In [31]:
def get_funnel(df, groupby):

    df_temp = df\
                .groupby(groupby)\
                .agg(responses = ('response_id', 'nunique'),
                     impression = ('impression_id', 'nunique'),
                     viewable_impression = ('viewable_impression_id', 'nunique'),
                     click = ('click_id', 'nunique'),
                    )\
                .reset_index()

    df_temp['% render rate'] = (df_temp['impression']*100.00/df_temp['responses']).round(2)
    df_temp['% view'] = (df_temp['viewable_impression']*100.00/df_temp['impression']).round(2)
    df_temp['% ctr'] = (df_temp['click']*100.00/df_temp['viewable_impression']).round(2)

    df_temp['format'] = format
    
    columns = ['format'] + groupby + ['responses', 'impression', 'viewable_impression', 'click', '% render rate', '% view', '% ctr']
    
    return df_temp[columns].sort_values(by=['Experiment', '% ctr'], ascending=[True, False])

<div class="alert alert-block alert-success">  
    
## Analysis
</div>

In [32]:
df_data.head(5)

,yyyymmdd,campaignName,slotUnitId,city,os,userid,service,orderstatus,response_id,impression_id,viewable_impression_id,click_id,gender,ltr_segment,retention_segments,regularity_segment,fe_regularity_segment,income_segment,service_affinity,gc_tag,Experiment
0,20250115,Exp1_MaxBanner_Zepto,PostOrderArrived3,Bangalore,android,5e5f1abf32e3140a0beaf16d,Auto,arrived,b57dbc1c-b86e-4519-a460-30a5478d9cd65e5f1abf32...,b57dbc1c-b86e-4519-a460-30a5478d9cd65e5f1abf32...,b57dbc1c-b86e-4519-a460-30a5478d9cd65e5f1abf32...,None,FEMALE,PHH,PLATINUM,DAILY,DAILY,MEDIUM_INCOME,ONLY_AUTO,UNKNOWN,Experiment-1
1,20250115,Exp1_MaxBanner_Zepto,PostOrderOnTheWay3,Bangalore,android,676c0d81a353a7da26899fd1,Auto,onTheWay,b8e7c852-d0f3-488f-af24-f3f415854597676c0d81a3...,b8e7c852-d0f3-488f-af24-f3f415854597676c0d81a3...,None,None,MALE,PHH,GOLD,BI_WEEKLY,MONTHLY,UNKNOWN,ONLY_LINK,UNKNOWN,Experiment-1
2,20250115,Exp1_MaxBanner_Zepto,PostOrderOnTheWay3,Kolkata,android,5d0a2fae1c64cc470f63ece1,Link,onTheWay,b07442b2-3166-4233-b884-90f6803d1bfd5d0a2fae1c...,b07442b2-3166-4233-b884-90f6803d1bfd5d0a2fae1c...,b07442b2-3166-4233-b884-90f6803d1bfd5d0a2fae1c...,None,MALE,PHH,PRIME,DAILY,DAILY,MEDIUM_INCOME,ONLY_LINK,NON_GC,Experiment-1
3,20250115,Exp1_MaxBanner_Zepto,PostOrderArrived3,Delhi,android,656adcfd044b093882cd1ff5,Bike Lite,arrived,849da0b9-fd51-4047-9d7d-6fcd709b820e656adcfd04...,849da0b9-fd51-4047-9d7d-6fcd709b820e656adcfd04...,849da0b9-fd51-4047-9d7d-6fcd709b820e656adcfd04...,None,FEMALE,PHH,ELITE,DAILY,DAILY,MEDIUM_INCOME,ONLY_LINK,UNKNOWN,Experiment-1
4,20250115,Exp1_MaxBanner_Zepto,PostOrderOnTheWay3,Chennai,android,6439178ecc7cb24c047ba197,Auto,onTheWay,a516d02e-d1f5-40e5-b957-a2e7251be96c6439178ecc...,a516d02e-d1f5-40e5-b957-a2e7251be96c6439178ecc...,a516d02e-d1f5-40e5-b957-a2e7251be96c6439178ecc...,None,FEMALE,PHH,GOLD,BI_WEEKLY,MONTHLY,LOW_INCOME,UNKNOWN,UNKNOWN,Experiment-1


#### Bivariate data 

In [33]:
get_funnel(df_data, ['Experiment', 'os']).pivot(index = ['Experiment'] , 
                                                columns = ['os'] , 
                                                values =['% render rate', '% view',	'% ctr', 'impression', 'viewable_impression', 'click'])

,% render rate,% view,% ctr,impression,viewable_impression,click
os,android,android,android,android,android,android
Experiment,,,,,,
Experiment-1,75.79,76.89,0.25,1176952.0,905003.0,2250.0
Experiment-2,79.59,75.15,0.03,1491501.0,1120818.0,375.0
Experiment-3,81.63,76.74,0.58,1637780.0,1256800.0,7297.0


In [18]:
display(get_funnel(df_data, ['Experiment', 'yyyymmdd']))

,format,Experiment,yyyymmdd,responses,% render rate,% view,% ctr
4,maxBanner,Experiment-1,20250119,29,96.55,82.14,4.35
5,maxBanner,Experiment-1,20250120,278302,84.01,83.41,0.75
0,maxBanner,Experiment-1,20250115,219413,81.52,79.73,0.17
6,maxBanner,Experiment-1,20250121,281593,72.69,74.86,0.10
7,maxBanner,Experiment-1,20250122,288297,72.64,75.25,0.10
2,maxBanner,Experiment-1,20250117,252570,72.17,73.32,0.09
1,maxBanner,Experiment-1,20250116,232757,72.12,73.21,0.08
3,maxBanner,Experiment-1,20250118,9,100.00,88.89,0.00
8,maxBanner,Experiment-2,20250123,263244,79.49,74.95,0.04
14,maxBanner,Experiment-2,20250129,278864,80.75,76.56,0.04


In [34]:
display(get_funnel(df_data, ['Experiment', 'slotUnitId']))

,format,Experiment,slotUnitId,responses,impression,viewable_impression,click,% render rate,% view,% ctr
2,maxBanner,Experiment-1,PostOrderStarted3,388903,308473,250117,1043,79.32,81.08,0.42
1,maxBanner,Experiment-1,PostOrderOnTheWay3,799900,579343,408202,865,72.43,70.46,0.21
0,maxBanner,Experiment-1,PostOrderArrived3,364167,289136,246684,342,79.40,85.32,0.14
5,maxBanner,Experiment-2,PostOrderStarted4,456807,360236,289376,160,78.86,80.33,0.06
4,maxBanner,Experiment-2,PostOrderOnTheWay4,977185,785984,542797,172,80.43,69.06,0.03
3,maxBanner,Experiment-2,PostOrderArrived4,439891,345281,288645,43,78.49,83.60,0.01
8,maxBanner,Experiment-3,PostOrderStarted1,492692,397748,324409,4018,80.73,81.56,1.24
6,maxBanner,Experiment-3,PostOrderArrived1,455180,362680,308560,1505,79.68,85.08,0.49
7,maxBanner,Experiment-3,PostOrderOnTheWay1,1058575,877352,623831,1774,82.88,71.10,0.28


In [20]:
display(get_funnel(df_data, ['Experiment', 'service']))

,format,Experiment,service,responses,% render rate,% view,% ctr
4,maxBanner,Experiment-1,Auto Pool,302,81.13,75.92,1.08
3,maxBanner,Experiment-1,Auto Pet,389,46.79,73.08,0.75
7,maxBanner,Experiment-1,Bike Pink,213,86.38,85.33,0.64
10,maxBanner,Experiment-1,CabPremium,29739,74.11,68.93,0.64
11,maxBanner,Experiment-1,E rickshaw,2237,82.12,79.70,0.48
5,maxBanner,Experiment-1,Bike Lite,102943,83.13,80.61,0.38
8,maxBanner,Experiment-1,C2C,12239,79.91,82.68,0.36
0,maxBanner,Experiment-1,Auto,570793,74.49,77.77,0.28
2,maxBanner,Experiment-1,Auto NCR,2265,81.02,78.64,0.28
6,maxBanner,Experiment-1,Bike Metro,28306,80.82,79.62,0.24


In [21]:
display(get_funnel(df_data, ['Experiment', 'gender']))

,format,Experiment,gender,responses,% render rate,% view,% ctr
3,maxBanner,Experiment-1,UNKNOWN,117418,78.34,74.87,0.35
2,maxBanner,Experiment-1,OTHERS,6570,79.22,72.70,0.34
0,maxBanner,Experiment-1,FEMALE,427815,76.32,78.03,0.29
1,maxBanner,Experiment-1,MALE,1001167,75.24,76.68,0.22
6,maxBanner,Experiment-2,OTHERS,8592,81.81,70.76,0.08
7,maxBanner,Experiment-2,UNKNOWN,131286,81.59,73.96,0.04
4,maxBanner,Experiment-2,FEMALE,434611,80.23,76.17,0.03
5,maxBanner,Experiment-2,MALE,1299394,79.16,74.95,0.03
11,maxBanner,Experiment-3,UNKNOWN,158018,83.11,74.70,0.78
10,maxBanner,Experiment-3,OTHERS,8183,84.81,72.90,0.75


In [22]:
display(get_funnel(df_data, ['Experiment', 'ltr_segment']))

,format,Experiment,ltr_segment,responses,% render rate,% view,% ctr
1,maxBanner,Experiment-1,NEW,84914,79.42,73.89,0.40
3,maxBanner,Experiment-1,UNKNOWN,21365,78.46,68.10,0.35
0,maxBanner,Experiment-1,HH,152742,77.67,75.00,0.30
2,maxBanner,Experiment-1,PHH,1293949,75.28,77.48,0.23
7,maxBanner,Experiment-2,UNKNOWN,28617,82.05,64.64,0.06
4,maxBanner,Experiment-2,HH,193557,80.56,73.21,0.05
5,maxBanner,Experiment-2,NEW,93298,82.28,73.07,0.04
6,maxBanner,Experiment-2,PHH,1558411,79.27,75.72,0.03
9,maxBanner,Experiment-3,NEW,118913,83.66,73.86,0.82
11,maxBanner,Experiment-3,UNKNOWN,34306,83.60,67.74,0.76


In [23]:
display(get_funnel(df_data, ['Experiment', 'retention_segments']))

,format,Experiment,retention_segments,responses,% render rate,% view,% ctr
8,maxBanner,Experiment-1,UNKNOWN,106279,79.23,72.74,0.39
4,maxBanner,Experiment-1,INACTIVE,88779,77.29,76.08,0.32
0,maxBanner,Experiment-1,DORMANT,209425,75.76,76.79,0.29
3,maxBanner,Experiment-1,HH,71293,77.63,74.91,0.25
5,maxBanner,Experiment-1,PLATINUM,298541,75.35,77.62,0.24
2,maxBanner,Experiment-1,GOLD,217031,75.69,77.23,0.23
7,maxBanner,Experiment-1,SILVER,105899,76.05,76.77,0.22
1,maxBanner,Experiment-1,ELITE,439886,74.70,77.82,0.21
6,maxBanner,Experiment-1,PRIME,15837,74.45,78.44,0.17
13,maxBanner,Experiment-2,INACTIVE,108800,79.96,74.03,0.05


In [24]:
display(get_funnel(df_data, ['Experiment', 'regularity_segment']))

,format,Experiment,regularity_segment,responses,% render rate,% view,% ctr
4,maxBanner,Experiment-1,UNKNOWN,259021,78.31,74.06,0.34
2,maxBanner,Experiment-1,MONTHLY,140102,75.93,77.45,0.26
3,maxBanner,Experiment-1,RARE_NEED,140347,76.13,77.37,0.26
0,maxBanner,Experiment-1,BI_WEEKLY,273190,75.75,77.21,0.24
5,maxBanner,Experiment-1,WEEKLY,374110,75.05,77.76,0.24
1,maxBanner,Experiment-1,DAILY,366200,74.60,77.46,0.20
9,maxBanner,Experiment-2,RARE_NEED,172596,79.39,75.75,0.05
10,maxBanner,Experiment-2,UNKNOWN,315472,81.20,72.38,0.05
8,maxBanner,Experiment-2,MONTHLY,172777,78.88,75.61,0.04
6,maxBanner,Experiment-2,BI_WEEKLY,339298,79.19,75.70,0.03


In [25]:
display(get_funnel(df_data, ['Experiment', 'fe_regularity_segment']))

,format,Experiment,fe_regularity_segment,responses,% render rate,% view,% ctr
4,maxBanner,Experiment-1,UNKNOWN,275021,77.61,75.72,0.33
2,maxBanner,Experiment-1,MONTHLY,200163,76.28,76.69,0.26
0,maxBanner,Experiment-1,BI_WEEKLY,357895,75.63,77.20,0.24
5,maxBanner,Experiment-1,WEEKLY,247981,75.20,77.61,0.24
3,maxBanner,Experiment-1,RARE_NEED,74406,75.93,75.66,0.23
1,maxBanner,Experiment-1,DAILY,397504,74.75,77.35,0.20
10,maxBanner,Experiment-2,UNKNOWN,323640,80.52,74.39,0.05
8,maxBanner,Experiment-2,MONTHLY,252649,79.46,74.77,0.04
9,maxBanner,Experiment-2,RARE_NEED,95309,79.81,74.02,0.04
6,maxBanner,Experiment-2,BI_WEEKLY,441893,79.27,75.40,0.03


In [26]:
display(get_funnel(df_data, ['Experiment', 'income_segment']))

,format,Experiment,income_segment,responses,% render rate,% view,% ctr
3,maxBanner,Experiment-1,UNKNOWN,360197,75.58,75.79,0.30
1,maxBanner,Experiment-1,LOW_INCOME,81883,81.89,78.13,0.28
2,maxBanner,Experiment-1,MEDIUM_INCOME,496059,80.77,78.26,0.26
0,maxBanner,Experiment-1,HIGH_INCOME,614831,71.08,76.14,0.20
5,maxBanner,Experiment-2,LOW_INCOME,105907,86.23,76.36,0.04
7,maxBanner,Experiment-2,UNKNOWN,434379,78.92,74.24,0.04
4,maxBanner,Experiment-2,HIGH_INCOME,731296,74.78,73.82,0.03
6,maxBanner,Experiment-2,MEDIUM_INCOME,602301,84.75,76.96,0.03
11,maxBanner,Experiment-3,UNKNOWN,503720,80.90,75.66,0.68
10,maxBanner,Experiment-3,MEDIUM_INCOME,637432,86.19,78.25,0.59


In [27]:
display(get_funnel(df_data, ['Experiment', 'service_affinity']))

,format,Experiment,service_affinity,responses,% render rate,% view,% ctr
7,maxBanner,Experiment-1,UNKNOWN,705023,76.66,76.23,0.28
4,maxBanner,Experiment-1,ONLY_AUTO,285642,72.65,78.44,0.27
0,maxBanner,Experiment-1,AUTO_CAB,22686,72.67,76.07,0.26
1,maxBanner,Experiment-1,LINK_AUTO,44260,74.67,78.40,0.22
3,maxBanner,Experiment-1,NO_AFFINITY,104799,72.96,76.42,0.21
6,maxBanner,Experiment-1,ONLY_LINK,319264,78.15,77.76,0.20
5,maxBanner,Experiment-1,ONLY_CAB,62700,74.46,73.50,0.19
2,maxBanner,Experiment-1,LINK_CAB,8596,79.11,74.09,0.14
15,maxBanner,Experiment-2,UNKNOWN,868344,79.69,74.66,0.04
11,maxBanner,Experiment-2,NO_AFFINITY,136414,79.12,74.52,0.03


In [28]:
display(get_funnel(df_data, ['Experiment', 'gc_tag']))

,format,Experiment,gc_tag,responses,% render rate,% view,% ctr
2,maxBanner,Experiment-1,UNKNOWN,1238371,75.95,76.75,0.26
1,maxBanner,Experiment-1,NON_GC,283779,75.02,77.35,0.20
0,maxBanner,Experiment-1,GC,30820,76.21,78.60,0.17
4,maxBanner,Experiment-2,NON_GC,347083,78.47,75.37,0.04
5,maxBanner,Experiment-2,UNKNOWN,1491294,79.85,75.07,0.03
3,maxBanner,Experiment-2,GC,35506,79.76,76.40,0.02
8,maxBanner,Experiment-3,UNKNOWN,1600628,81.84,76.57,0.60
7,maxBanner,Experiment-3,NON_GC,373794,80.73,77.30,0.50
6,maxBanner,Experiment-3,GC,32025,81.65,78.38,0.43


#### Multivariate data 

In [29]:
get_funnel(df_data, ['Experiment', 'slotUnitId', 'os']).pivot(index = ['slotUnitId', 'Experiment'] , 
                                                columns = ['os'] , 
                                                values =['% render rate', '% view',	'% ctr'])

,,% render rate,% view,% ctr
,os,android,android,android
slotUnitId,Experiment,,,
PostOrderArrived1,Experiment-3,79.68,85.08,0.49
PostOrderArrived3,Experiment-1,79.40,85.32,0.14
PostOrderArrived4,Experiment-2,78.49,83.60,0.01
PostOrderOnTheWay1,Experiment-3,82.88,71.10,0.28
PostOrderOnTheWay3,Experiment-1,72.43,70.46,0.21
PostOrderOnTheWay4,Experiment-2,80.43,69.06,0.03
PostOrderStarted1,Experiment-3,80.73,81.56,1.24
PostOrderStarted3,Experiment-1,79.32,81.08,0.42


In [30]:
display(get_funnel(df_data, ['Experiment', 'yyyymmdd', 'slotUnitId']))

,format,Experiment,yyyymmdd,slotUnitId,responses,% render rate,% view,% ctr
13,maxBanner,Experiment-1,20250119,PostOrderOnTheWay3,17,100.00,88.24,6.67
17,maxBanner,Experiment-1,20250120,PostOrderStarted3,75653,82.96,86.44,1.21
16,maxBanner,Experiment-1,20250120,PostOrderOnTheWay3,136172,85.24,77.87,0.65
15,maxBanner,Experiment-1,20250120,PostOrderArrived3,66477,82.69,91.65,0.45
2,maxBanner,Experiment-1,20250115,PostOrderStarted3,55374,80.59,84.04,0.31
20,maxBanner,Experiment-1,20250121,PostOrderStarted3,68826,78.43,79.62,0.19
23,maxBanner,Experiment-1,20250122,PostOrderStarted3,70619,78.26,79.53,0.18
8,maxBanner,Experiment-1,20250117,PostOrderStarted3,61748,77.62,77.57,0.16
1,maxBanner,Experiment-1,20250115,PostOrderOnTheWay3,113140,82.61,73.45,0.14
5,maxBanner,Experiment-1,20250116,PostOrderStarted3,56671,77.45,78.00,0.13
